# Data Extraction Flow — Mining and Oil (ANM CFEM, AMB, SCM, SIGBM, DIPEM + ANP wells)

| branch | module | notes |
|---|---|---|
| ANM CFEM | `pipelines.cfem` | single pass, `lookback_meses=6` |
| ANM AMB | `pipelines.amb` | yearly, single pass |
| ANP production by well | `pipelines.anp` | *while loop* until `has_more=False`, 12 months per iteration |
| ANM Mining Registry (SCM) | `pipelines.scm` | snapshot; `range` runs as `incremental` |
| ANM SIGBM dams | `pipelines.sigbm` | snapshot |
| ANM DIPEM investment | `pipelines.dipem` | snapshot |

In [1]:
from pprint import pprint
import pandas as pd
from pipelines import common as C
from pipelines.flow import run_flow

pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 60)

## Inputs

In [2]:
mode = "incremental"      # incremental | range | backfill
reference_date = ""         # YYYY-MM-DD; empty = today (America/Sao_Paulo)
start_date = ""             # YYYY-MM or YYYY-MM-DD (required for range; empty backfill = 2015-01)
end_date = ""
force = False               # reload even if ETag/Last-Modified did not change
target_db = "bronze"        # target prefix in the lake bucket (use another name for tests)
cfg_db = ""                 # where dim_substanciaMineral / dim_alvoExtracao live (Parquet); empty = code defaults

# study extras
dry_run = True              # True = plan + light source check only (HEAD + header)
parallel = False            # True = branches in threads (only 4 cores here, so sequential by default)

## Execute

In [3]:
spark = None if dry_run else C.get_spark("flow_mineracao_petroleo", cores_max=4)
resumo = run_flow(spark, mode=mode, reference_date=reference_date, start_date=start_date, end_date=end_date,
                  force=force, target_db=target_db, cfg_db=cfg_db, dry_run=dry_run, parallel=parallel)
pd.DataFrame(resumo)

[CFEM] {'fonte': 'anm_cfem', 'mode': 'incremental', 'janela': '2015-01..2026-10', 'watermark': None, 'urls': ['https://dadosabertos.anm.gov.br/CFEM/CFEM_Arrecadacao.csv'], 'destino': 'bronze.anm_cfem_arrecadacao', 'foco': {'origem': 'padrao_script', 'substancias': ['BAUXITA', 'COBRE', 'FERRO', 'FOSFATO', 'MANGANES', 'NIOBIO', 'NIQUEL', 'OURO'], 'ufs': ['AL', 'BA', 'ES', 'GO', 'MA', 'MG', 'PA', 'RJ', 'SE', 'SP']}}


[AMB] {'fonte': 'anm_amb', 'mode': 'incremental', 'janela': '2010..2026', 'urls': ['https://dadosabertos.anm.gov.br/AMB/Producao_Bruta.csv', 'https://dadosabertos.anm.gov.br/AMB/Producao_Beneficiada.csv'], 'destino': 'bronze.anm_amb_producao', 'foco': {'origem': 'padrao_script', 'substancias': ['BAUXITA', 'COBRE', 'FERRO', 'FOSFATO', 'MANGANES', 'NIOBIO', 'NIQUEL', 'OURO'], 'ufs': ['AL', 'BA', 'ES', 'GO', 'MA', 'MG', 'PA', 'RJ', 'SE', 'SP']}}


[ANP] {'fonte': 'anp_poco', 'mode': 'incremental', 'janela': '2026-08..2026-10', 'lote': '2026-08..2026-10', 'watermark': None, 'destino': 'bronze.anp_producao_poco', 'bacias': ['CAMPOS', 'SANTOS', 'SERGIPE', 'ALAGOAS', 'ESPIRITO SANTO'], 'links_encontrados': 52, 'links_com_periodo': 52, 'cobertura_links': '2005-01..2023-12', 'arquivos_do_lote': [], 'has_more': False, 'aviso': "a janela termina em 2026-10 mas os links da página cobrem só até 2023-12 — nenhum arquivo será lido e a carga sai 'ok' com 0 linhas (a ANP publica 2024 em diante só na Consulta de Dados Públicos, sem links de arquivo)"}
[SCM] {'fonte': 'anm_scm', 'mode': 'incremental', 'url': 'https://dadosabertos.anm.gov.br/SCM/microdados/microdados-scm.zip', 'destino': ['bronze.anm_scm_processo', 'bronze.anm_scm_substancia', 'bronze.anm_scm_municipio', 'bronze.anm_scm_pessoa', 'bronze.anm_scm_titulo', 'bronze.anm_scm_evento'], 'foco': {'origem': 'padrao_script', 'substancias': ['BAUXITA', 'COBRE', 'FERRO', 'FOSFATO', 'MANGANES

[SIGBM] {'fonte': 'anm_sigbm', 'mode': 'incremental', 'urls': ['https://dadosabertos.anm.gov.br/SIGBM/Barragens.csv'], 'destino': 'bronze.anm_sigbm_barragem', 'foco': {'origem': 'padrao_script', 'substancias': ['BAUXITA', 'COBRE', 'FERRO', 'FOSFATO', 'MANGANES', 'NIOBIO', 'NIQUEL', 'OURO'], 'ufs': ['AL', 'BA', 'ES', 'GO', 'MA', 'MG', 'PA', 'RJ', 'SE', 'SP']}}


[DIPEM] {'fonte': 'anm_dipem', 'mode': 'incremental', 'urls': ['https://dadosabertos.anm.gov.br/DIPEM/InvestimentoPesquisaMineralMunicipio.csv'], 'destino': 'bronze.anm_dipem_investimento', 'foco': {'origem': 'padrao_script', 'substancias': ['BAUXITA', 'COBRE', 'FERRO', 'FOSFATO', 'MANGANES', 'NIOBIO', 'NIQUEL', 'OURO'], 'ufs': ['AL', 'BA', 'ES', 'GO', 'MA', 'MG', 'PA', 'RJ', 'SE', 'SP']}}


,ramo,fonte,mode,status,janela,linhas,cursor,has_more,fonte_ok,aviso
0,ANM CFEM,anm_cfem,incremental,dry_run,2015-01..2026-10,None,None,False,ok,NaN
1,ANM AMB,anm_amb,incremental,dry_run,2010..2026,None,None,False,ok,NaN
2,ANP produção por poço,anp_poco,incremental,dry_run,2026-08..2026-10,None,None,False,0 arquivo(s) no lote,a janela termina em 2026-10 mas os links da página cobre...
3,ANM Cadastro Mineiro (SCM),anm_scm,incremental,dry_run,None,None,None,False,ok,NaN
4,ANM SIGBM barragens,anm_sigbm,incremental,dry_run,None,None,None,False,ok,NaN
5,ANM DIPEM investimento em pesquisa,anm_dipem,incremental,dry_run,None,None,None,False,ok,NaN


## Source check (dry run only)

Detailed result of each branch (logs above). For a single source, use its own notebook.